In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
import io


In [2]:
from google.colab import files
uploaded = files.upload()

Saving detect.csv to detect (1).csv


In [5]:
df = pd.read_csv(io.StringIO(uploaded["detect (1).csv"].decode('utf-8')))

print(df.head())


   sport dport proto state       dur  sbytes  spkts     label
0   1390    53   udp   CON  0.001055     132      2       NaN
1  33661  1024   udp   CON  0.036133     528      4       NaN
2   1464    53   udp   CON  0.001119     146      2  Exploits
3   3593    53   udp   CON  0.001209     132      2       NaN
4  49664    53   udp   CON  0.001169     146      2       NaN


In [6]:
# Replace NaN with 0, otherwise replace with 1 in the 'label' column
df['label'] = df['label'].apply(lambda x: 0 if pd.isna(x) else 1)


In [7]:
print(df.head())

   sport dport proto state       dur  sbytes  spkts  label
0   1390    53   udp   CON  0.001055     132      2      0
1  33661  1024   udp   CON  0.036133     528      4      0
2   1464    53   udp   CON  0.001119     146      2      1
3   3593    53   udp   CON  0.001209     132      2      0
4  49664    53   udp   CON  0.001169     146      2      0


In [8]:
le = LabelEncoder()

# Encode categorical columns
df['sport'] = le.fit_transform(df['sport'])
df['dport'] = le.fit_transform(df['dport'])
df['proto'] = le.fit_transform(df['proto'])
df['state'] = le.fit_transform(df['state'])
df['label'] = le.fit_transform(df['label'])
print(df.head())

   sport  dport  proto  state       dur  sbytes  spkts  label
0   3278  16443    120      2  0.001055     132      2      0
1  19641     86    120      2  0.036133     528      4      0
2   3893  16443    120      2  0.001119     146      2      1
3  21476  16443    120      2  0.001209     132      2      0
4  32737  16443    120      2  0.001169     146      2      0


In [9]:
# Features
X = df.drop(columns=['label'])

# Target variable
y = df['label']


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
smote = SMOTE(random_state=42)

# Apply SMOTE to balance the dataset
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE, class distribution in y_train: {y_train.value_counts()}")
print(f"After SMOTE, class distribution in y_train: {y_train_res.value_counts()}")


Before SMOTE, class distribution in y_train: label
0    70888
1     9112
Name: count, dtype: int64
After SMOTE, class distribution in y_train: label
0    70888
1    70888
Name: count, dtype: int64


In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Fit the scaler to the training data and transform both training and test sets
X_train_res = scaler.fit_transform(X_train_res)
X_test = scaler.transform(X_test)

print(X_train_res[:5])


[[ 0.70813067 -0.45390371  0.13104754  0.01412937 -0.21034992 -0.03489687
   0.27115278]
 [ 0.84602366 -0.34742952  0.13104754  0.01412937 -0.21298536 -0.03847306
   0.19501447]
 [-0.22741439 -0.54462656  0.13104754  0.01412937  0.05536439  1.00144402
   0.86122469]
 [ 0.41522608  0.35393258  0.13104754  0.01412937  0.17417413 -0.0548308
  -0.14760793]
 [ 0.66948791  0.73098798  0.44550307  1.05259802 -0.22251756 -0.06077456
  -0.24278082]]


In [19]:
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

In [20]:
classifiers = {
    "SGDClassifier": SGDClassifier(random_state=42),
    "SVM": LinearSVC(random_state=42),
    "KNN": KNeighborsClassifier(),
    "GaussianNB": GaussianNB(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

In [21]:
results = {}

for name, clf in classifiers.items():
    print(f"\n--- {name} ---")
    clf.fit(X_train_res, y_train_res)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')  # or 'macro'/'micro'

    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    results[name] = {
        "accuracy": acc,
        "f1_score": f1,
    }


--- SGDClassifier ---
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.85      0.91     17681
           1       0.41      0.81      0.55      2319

    accuracy                           0.84     20000
   macro avg       0.69      0.83      0.73     20000
weighted avg       0.91      0.84      0.86     20000

Confusion Matrix:
[[15013  2668]
 [  438  1881]]

--- SVM ---
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.84      0.90     17681
           1       0.40      0.83      0.54      2319

    accuracy                           0.84     20000
   macro avg       0.69      0.83      0.72     20000
weighted avg       0.91      0.84      0.86     20000

Confusion Matrix:
[[14845  2836]
 [  405  1914]]

--- KNN ---
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.98     17681
           1       0.

In [22]:
print("\n--- Model Comparison ---")
for model, metrics in results.items():
    print(f"{model} -> Accuracy: {metrics['accuracy']:.4f}, F1 Score: {metrics['f1_score']:.4f}")


--- Model Comparison ---
SGDClassifier -> Accuracy: 0.8447, F1 Score: 0.8647
SVM -> Accuracy: 0.8379, F1 Score: 0.8598
KNN -> Accuracy: 0.9622, F1 Score: 0.9638
GaussianNB -> Accuracy: 0.8388, F1 Score: 0.8272
DecisionTree -> Accuracy: 0.9829, F1 Score: 0.9830
RandomForest -> Accuracy: 0.9830, F1 Score: 0.9834
AdaBoost -> Accuracy: 0.9237, F1 Score: 0.9307
